# 02 - Modeling (the core notebook)

Pipeline: **data -> features -> target -> chronological split -> baselines -> tuned models -> validation comparison ->
refit -> single test evaluation -> explainability -> saved predictions.**

Rules enforced throughout: no shuffling, no random `train_test_split`, preprocessing fit inside the sklearn `Pipeline`,
hyper-parameters tuned **only on train** with an expanding-window `TimeSeriesSplit(gap=horizon)`, model *selected* on validation,
test set scored **once**.

In [ ]:
%matplotlib inline
import os, sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 140); pd.set_option("display.max_columns", 40)

from spy_signals import Config, plotting as P
from spy_signals.data import load_ohlcv

# SPY_FAST=1 shrinks the hyper-parameter grids; SPY_DATA_SOURCE=synthetic runs offline (smoke test only!)
cfg = Config(fast=os.environ.get("SPY_FAST") == "1")
ohlcv = load_ohlcv(cfg)

## 1. Data, features, target, chronological split

In [ ]:
from spy_signals.pipeline import prepare_data
from spy_signals.splits import chronological_split
ds = prepare_data(cfg, ohlcv)
splits = chronological_split(ds.X, ds.y, cfg)
splits.summary()

In [ ]:
from spy_signals import audit
from spy_signals.models import time_series_cv
audit.format_audits([
    audit.audit_no_lookahead(ohlcv),
    audit.audit_target_not_in_features(ds.X, ds.fwd_ret),
    audit.audit_split_order(ds.X, splits, ohlcv.index, cfg.horizon),
    audit.audit_no_shuffle_cv(time_series_cv(cfg), splits.X_train),
])

## 2. Baselines first
Any model must beat these, otherwise the simple rule *is* the answer (same rule as the L06 forecasting checklist).

In [ ]:
from spy_signals import evaluate, models
val_rows = {}
for name, est in models.baseline_models().items():
    est.fit(splits.X_train, splits.y_train)
    val_rows[name] = evaluate.classification_summary(splits.y_val, est.predict(splits.X_val))
evaluate.comparison_table(val_rows)

## 3. Models and hyper-parameter tuning
Three interpretable tabular classifiers, in increasing complexity: **Logistic Regression** (linear, scaled), **Random Forest**,
**Gradient Boosting** (`HistGradientBoostingClassifier`; swap in XGBoost/LightGBM if you prefer). All use **class weights = balanced**
so Sell/Hold/Buy count equally in the loss (assignment requirement: handle imbalance).

Tuning = `GridSearchCV` over `TimeSeriesSplit(n_splits=5, gap=horizon)` scored by **macro-F1**:
each fold trains on the past and validates on the *next* block, never the reverse.

In [ ]:
zoo = models.model_zoo(cfg)
fitted = {}
for name, (est, grid) in zoo.items():
    search = models.tune(name, est, grid, splits.X_train, splits.y_train, cfg)
    fitted[name] = search
    val_rows[name] = evaluate.classification_summary(splits.y_val, search.predict(splits.X_val))
    print(f"{name:14s} CV macro-F1 = {search.best_score_:.4f}   best params: {search.best_params_}")

In [ ]:
# What did the tuning surface look like? (flat surfaces = hyper-parameters barely matter on noisy data)
for name, s in fitted.items():
    cv = pd.DataFrame(s.cv_results_)[["params", "mean_test_score", "std_test_score"]].sort_values("mean_test_score", ascending=False)
    print(f"--- {name} ---"); display(cv.head(5).round(4))

## 4. Compare on the validation period (2021-2022 by default) and select

In [ ]:
cmp = evaluate.comparison_table(val_rows)
display(cmp)
best_name = max(fitted, key=lambda m: val_rows[m]["macro_f1"])
print("Selected on validation macro-F1:", best_name)
P.plot_confusion(val_rows[best_name]["confusion"], f"Validation confusion - {best_name}");

**How to read it:** ~0.33 is chance for 3 classes; the majority baseline can have higher *accuracy* but near-zero *balanced accuracy*.
If your validation accuracy is far above ~45 %, stop and look for a leak.

## 5. Explainability (on validation, with the train-only model - the test set is still untouched)

In [ ]:
from spy_signals import explain
train_only = fitted[best_name].best_estimator_
perm = explain.permutation_table(train_only, splits.X_val, splits.y_val, cfg.scoring, seed=cfg.random_state)
display(perm.head(10).round(4))
P.plot_importance(perm);

### Global view of the *linear* model (signed coefficients per class)
Coefficients are on standardised features, so sizes are comparable. Positive in the **Buy** column = pushes toward Buy.

In [ ]:
lr = fitted["logreg"].best_estimator_
coefs = explain.logreg_coefficients(lr, ds.X.columns)
display(coefs.round(3).loc[coefs.abs().max(axis=1).sort_values(ascending=False).index].head(10))
P.plot_logreg_coefs(coefs);

### Local explanation: why did the logistic model say what it said on one specific day?
`contribution = coefficient x standardised feature value`; the sum plus the intercept is the class score.

In [ ]:
day = splits.X_val.index[len(splits.X_val) // 2]
tbl = explain.explain_prediction_logreg(lr, splits.X_val.loc[[day]])
print(f"{day.date()}: logistic regression predicts {tbl.attrs['predicted']} (actual: {evaluate.NAMES[int(splits.y_val.loc[day])]})")
tbl.round(4)

## 6. Final fit and the one-time test evaluation
Refit the selected configuration on **train + validation** (purged at the boundary), then score the untouched test period once.

In [ ]:
from sklearn.base import clone
import joblib
final = clone(fitted[best_name].best_estimator_).fit(splits.X_trainval, splits.y_trainval)
test_pred = final.predict(splits.X_test)
test = evaluate.classification_summary(splits.y_test, test_pred)
maj = evaluate.majority_accuracy(splits.y_trainval, splits.y_test)
print(f"TEST accuracy {test['accuracy']:.3f} | majority-class baseline {maj:.3f} | chance 0.333")
print(f"balanced accuracy {test['balanced_accuracy']:.3f} | macro-F1 {test['macro_f1']:.3f}")
display(pd.DataFrame(test["per_class"]).T.round(3))
P.plot_confusion(test["confusion"], f"TEST confusion - {best_name}");

In [ ]:
# Live signals for EVERY test date (including the last `horizon` days, which have features but no label yet)
X_live = ds.X_all.loc[ds.X_all.index > pd.Timestamp(cfg.val_end)]
proba = pd.DataFrame(final.predict_proba(X_live), index=X_live.index,
                     columns=[evaluate.NAMES[c] for c in final.named_steps["clf"].classes_])
predictions = proba.assign(pred=final.predict(X_live), confidence=proba.max(axis=1))
out = Path(cfg.output_dir); out.mkdir(parents=True, exist_ok=True)
predictions.to_csv(out / "test_predictions.csv"); joblib.dump(final, out / "final_model.joblib")
print("saved:", out / "test_predictions.csv"); predictions.tail()

## 7. Sanity / honesty checks
* Accuracy near 35-40 %? Normal. Near 80 %? A leak - re-run the audits above.
* Did the model beat the majority baseline on **balanced accuracy**? If not, there is no evidence of skill.
* Validation vs test gap = how fragile the selection was. Next notebook: does any of this survive **trading costs**?